# Notebook 02 — PyTorch Fundamentals

**Module:** 14 — Deep Learning and GNNs  
**Tier:** 2 — Working competence  
**Notebook:** 2 of 12  
**Time estimate:** 75 minutes

> PyTorch is the framework of choice for computational biology research.
> The goal here is not to memorize the API — it's to understand what autograd
> is doing under the hood so you can debug it when it breaks.

---
## Step 1 — Motivation

The NB01 MLP works, but every new architecture requires re-deriving backprop.
PyTorch's `autograd` engine tracks every operation on tensors and computes
gradients automatically. You only write the forward pass; PyTorch writes the
backward pass. This is the fundamental shift from classical ML to deep learning practice.

---
## Step 2 — Intuition

**Tensor:** an n-dimensional array, like numpy's `ndarray`, but with:
- GPU support (`tensor.to('cuda')`)
- Gradient tracking (`requires_grad=True`)
- A computation graph built implicitly during forward pass

**Autograd:** every tensor operation records itself in a directed acyclic graph
(computation graph). Calling `loss.backward()` traverses this graph in reverse,
applying the chain rule to accumulate gradients in `.grad` attributes.

**`nn.Module`:** the building block for all PyTorch models.
- `__init__`: define layers (they register parameters automatically)
- `forward(x)`: define the computation — this is the forward pass
- `.parameters()`: iterator over all learnable parameters

**`DataLoader`:** wraps a `Dataset` and handles batching, shuffling, parallel loading.
Biological datasets are always too large to fit in one batch — DataLoader handles this.

**The training loop (every PyTorch model uses this pattern):**
```python
for epoch in range(n_epochs):
    for X_batch, y_batch in dataloader:
        optimizer.zero_grad()     # clear accumulated gradients
        y_pred = model(X_batch)   # forward pass
        loss = criterion(y_pred, y_batch)
        loss.backward()           # backprop
        optimizer.step()          # update parameters
```

---
## Step 3 — Biological Background

**GPU acceleration matters for biology:**
- AlphaFold2 on a 400-residue protein: ~10 min on GPU, hours on CPU
- scVI on 100k cells: GPU reduces training from hours to minutes
- Even sequence CNNs benefit: 10x speedup on genomic datasets

**Batch size in biology:**
Biological datasets are often small (hundreds to thousands of samples).
Batch size of 32–128 is common. Smaller batches = noisier gradients but
better generalization (implicit regularization).

**Reproducibility with PyTorch:**
Set seeds: `torch.manual_seed(42)`, `np.random.seed(42)`.
CUDA operations are non-deterministic by default — use `torch.use_deterministic_algorithms(True)`
for reproducible benchmarks (with a performance cost).

**`torch.no_grad()` context manager:**
During inference and evaluation, disable gradient computation:
saves memory and is ~2x faster. Always use for evaluation loops.

---
## Step 4 — Mathematical Explanation

**Computation graph:** PyTorch represents the forward pass as a DAG of
`Function` objects. Each node stores the local Jacobian (or enough information
to compute it). `.backward()` performs reverse-mode automatic differentiation:
accumulates $\partial \mathcal{L} / \partial \theta$ for each parameter $\theta$.

**Why reverse-mode?** For a scalar loss and many parameters:
- Forward-mode AD: one pass per parameter (expensive for large models)
- Reverse-mode AD: one backward pass for all parameters simultaneously

**`optimizer.zero_grad()` is critical:**
PyTorch accumulates gradients by default (`.grad += new_grad`).
Calling `zero_grad()` before each backward pass prevents gradient accumulation
across batches. Forgetting this is one of the most common PyTorch bugs.

**Adam optimizer:**
$$m_t = \beta_1 m_{t-1} + (1-\beta_1)g_t \quad \text{(first moment, momentum)}$$
$$v_t = \beta_2 v_{t-1} + (1-\beta_2)g_t^2 \quad \text{(second moment, RMS)}$$
$$\hat{m}_t = m_t/(1-\beta_1^t), \quad \hat{v}_t = v_t/(1-\beta_2^t) \quad \text{(bias correction)}$$
$$\theta_{t+1} = \theta_t - \eta \hat{m}_t / (\sqrt{\hat{v}_t} + \epsilon)$$

Defaults: $\beta_1=0.9$, $\beta_2=0.999$, $\epsilon=10^{-8}$, $\eta=10^{-3}$.

In [ ]:
# Step 6 — Python: PyTorch fundamentals + port of NB01 MLP
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

# ---- 1. Autograd demonstration ----
x = torch.tensor(3.0, requires_grad=True)
y = x**2 + 2*x + 1  # f(x) = x^2 + 2x + 1
y.backward()
print(f'\nAutograd demo: f(x) = x^2 + 2x + 1 at x=3')
print(f'  f(3) = {y.item():.1f}')
print(f'  f\'(3) = {x.grad.item():.1f}  (analytic: 2*3+2 = 8)')

# ---- 2. Port NB01 MLP to PyTorch ----
class MLPTorch(nn.Module):
    def __init__(self, layer_sizes):
        super().__init__()
        layers = []
        for i in range(len(layer_sizes) - 2):
            layers.append(nn.Linear(layer_sizes[i], layer_sizes[i+1]))
            layers.append(nn.ReLU())
        layers.append(nn.Linear(layer_sizes[-2], layer_sizes[-1]))
        layers.append(nn.Sigmoid())
        self.net = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.net(x)

# XOR problem — should match NB01
X_xor = torch.tensor([[0,0],[0,1],[1,0],[1,1]], dtype=torch.float32)
y_xor = torch.tensor([[0],[1],[1],[0]], dtype=torch.float32)

model_xor = MLPTorch([2, 4, 1]).to(device)
optimizer_xor = optim.SGD(model_xor.parameters(), lr=0.5)
criterion = nn.BCELoss()

loss_hist_torch = []
for epoch in range(2000):
    optimizer_xor.zero_grad()
    y_pred = model_xor(X_xor.to(device))
    loss = criterion(y_pred, y_xor.to(device))
    loss.backward()
    optimizer_xor.step()
    loss_hist_torch.append(loss.item())
print(f'\nXOR (PyTorch): final loss={loss_hist_torch[-1]:.4f}')
with torch.no_grad():
    preds = model_xor(X_xor.to(device)).cpu().numpy().ravel()
print(f'  Predictions: {preds.round(3)}')

# ---- 3. DataLoader pattern ----
class BiologyDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Enhancer dataset
rng = np.random.default_rng(42)
n = 500
X_enh = np.vstack([rng.normal([2.5, 2.8, 2.2, 0.3, 0.5, 0.8, 1.2, 0.4, 0.6, 1.0], 0.5, (n//2, 10)),
                     rng.normal([0.3, 0.4, 0.5, 2.5, 1.8, 0.2, 0.3, 1.5, 0.8, 0.4], 0.5, (n//2, 10))])
y_enh = np.array([1]*(n//2) + [0]*(n//2), dtype=float)

dataset = BiologyDataset(X_enh, y_enh)
train_size = int(0.8 * len(dataset))
train_ds, val_ds = torch.utils.data.random_split(dataset, [train_size, len(dataset)-train_size])
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64)

model_enh = MLPTorch([10, 32, 16, 1]).to(device)
optimizer_enh = optim.Adam(model_enh.parameters(), lr=1e-3)

train_losses, val_losses = [], []
for epoch in range(100):
    model_enh.train()
    batch_losses = []
    for X_b, y_b in train_loader:
        optimizer_enh.zero_grad()
        pred = model_enh(X_b.to(device))
        loss = criterion(pred, y_b.to(device))
        loss.backward()
        optimizer_enh.step()
        batch_losses.append(loss.item())
    train_losses.append(np.mean(batch_losses))
    
    model_enh.eval()
    with torch.no_grad():
        val_bl = [criterion(model_enh(X_b.to(device)), y_b.to(device)).item()
                   for X_b, y_b in val_loader]
    val_losses.append(np.mean(val_bl))

print(f'\nEnhancer (PyTorch+DataLoader): final val loss={val_losses[-1]:.4f}')

# ---- 4. Model introspection ----
print(f'\nModel architecture:')
print(model_enh)
print(f'\nParameter count:')
total = sum(p.numel() for p in model_enh.parameters())
train_p = sum(p.numel() for p in model_enh.parameters() if p.requires_grad)
print(f'  Total: {total}')
print(f'  Trainable: {train_p}')

In [ ]:
# Step 7 — Visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Panel A: numpy vs. PyTorch loss curve on XOR
ax = axes[0]
ax.plot(range(1, 2001), loss_hist_torch, 'steelblue', lw=1.5, label='PyTorch (SGD lr=0.5)')
ax.set_xlabel('Epoch'); ax.set_ylabel('BCE Loss')
ax.set_title('A. XOR training loss\n(PyTorch autograd)')
ax.legend(fontsize=9); ax.set_yscale('log')

# Panel B: Train vs. validation loss (enhancer)
ax = axes[1]
ax.plot(train_losses, 'steelblue', lw=1.5, label='Train')
ax.plot(val_losses, 'tomato', lw=1.5, label='Validation')
ax.set_xlabel('Epoch'); ax.set_ylabel('BCE Loss')
ax.set_title('B. Train/val loss (enhancer)\n(mini-batch SGD with DataLoader)')
ax.legend(fontsize=9)

# Panel C: Computation graph visualization (toy example)
ax = axes[2]
ax.axis('off')
# Draw a simple computation graph manually
nodes = {
    'x1': (0.1, 0.5), 'x2': (0.1, 0.2),
    'z1': (0.35, 0.7), 'z2': (0.35, 0.3),
    'a1': (0.55, 0.7), 'a2': (0.55, 0.3),
    'z_out': (0.75, 0.5), 'y_hat': (0.9, 0.5),
}
edges = [('x1','z1'),('x2','z1'),('x1','z2'),('x2','z2'),
           ('z1','a1'),('z2','a2'),('a1','z_out'),('a2','z_out'),('z_out','y_hat')]
for (x,y) in nodes.values():
    ax.add_patch(plt.Circle((x,y), 0.04, color='steelblue', zorder=3))
for (n1, n2) in edges:
    x1,y1 = nodes[n1]; x2,y2 = nodes[n2]
    ax.annotate('', xy=(x2,y2), xytext=(x1,y1),
                   arrowprops=dict(arrowstyle='->', color='grey', lw=1.5))
for name, (x,y) in nodes.items():
    ax.text(x, y+0.07, name, ha='center', fontsize=9, fontweight='bold')
ax.text(0.5, 0.05, 'Forward →\nBackward (autograd) ←', ha='center', fontsize=10, style='italic')
ax.set_title('C. Computation graph\n(autograd traverses this in reverse)')
ax.set_xlim(0, 1); ax.set_ylim(0, 1)

plt.suptitle('Module 14 NB02: PyTorch Fundamentals', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('pytorch_fundamentals.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Exercises

1. Implement the NB01 numpy MLP and the NB02 PyTorch MLP with identical
   initialization and learning rate. Train both for 1000 epochs on XOR.
   Show that the loss curves are identical.
2. Explain what `optimizer.zero_grad()` does. Show by example what happens to
   training if you remove it.
3. Implement a custom `Dataset` for a CSV file of biological features.
   Show that a `DataLoader` with `batch_size=32, shuffle=True` produces different
   batches each epoch.
4. Count the parameters in the enhancer model manually (from the `layer_sizes` list).
   Verify your count matches `sum(p.numel() for p in model.parameters())`.

---
## Step 10 — Quiz

1. What does `requires_grad=True` on a tensor do?
2. Why does PyTorch need `optimizer.zero_grad()` before each backward pass?
3. What is the difference between `model.train()` and `model.eval()`?
4. Write the Adam update rule. What do the first and second moment estimates represent?
5. What is `torch.no_grad()` and when should you use it?

---
## Step 12 — Reflection

> *[In one paragraph: explain what PyTorch's autograd is computing and how it
> differs from symbolic differentiation (e.g., as in SymPy). What advantage
> does define-by-run (PyTorch) have over define-then-run (TensorFlow 1.x) for
> biological research?]*

---
*Next: `03_cnn_for_sequence_classification.ipynb`*